# 📱 SMS 카테고리 분류 모델
KLUE/BERT 기반 SMS 카테고리 분류

**이 노트북의 목표**: 문자 텍스트를 7개 카테고리로 분류

PERSONAL / FINANCE / DELIVERY / GOVERNMENT / PROMOTION / AUTH / WORK

순수 문맥 분석 기반 카테고리 분류만 수행합니다.

## 1. 패키지 설치

In [ ]:
!pip install transformers torch scikit-learn pandas tqdm -q

## 2. 설정

In [ ]:
CATEGORIES = {
    0: 'PERSONAL',
    1: 'FINANCE',
    2: 'DELIVERY',
    3: 'GOVERNMENT',
    4: 'PROMOTION',
    5: 'AUTH',
    6: 'WORK',
}

MODEL_NAME       = 'klue/bert-base'
NUM_LABELS       = len(CATEGORIES)
MAX_LENGTH       = 128
BATCH_SIZE       = 32
EVAL_BATCH_SIZE  = 64
GRAD_ACCUM_STEPS = 2
EPOCHS           = 12
LR               = 2e-5
DROPOUT          = 0.3
LABEL_SMOOTHING  = 0.1
FOCAL_GAMMA      = 1.5
SAVE_PATH        = '/content/best_model.pt'
SEED             = 42
TEST_SIZE        = 0.2
VAL_SIZE         = 0.1
NUM_WORKERS      = 2
PIN_MEMORY       = True

KEYWORD_BOOSTS = {
    0: {  # PERSONAL
        'keywords': [
            # 가족 관계 (강화)
            '엄마', '아빠', '아버지', '어머니',
            '오빠', '언니', '형', '누나', '동생',
            '할머니', '할아버지', '할머', '할아버',
            '손주', '손녀', '손자',
            '아들', '딸', '아내', '남편',
            # 친교 관계
            '친구', '친구들', '친한친구', '절친',
            '자기', '여보', '자기야',
            # 감정 표현
            '사랑해', '보고싶어', '그리워', '싶어',
            '미안', '죄송', '잘못했어', '잘못',
            '고마워', '감사', '고마움',
            '축하', '축하해', '생일축하',
            # 만남/약속 관련
            '만나자', '같이', '함께', '같이가', '함께가',
            '봐', '뵐게', '만나', '만날래',
            '놀러', '여행', '여행가',
            # 일상 활동
            '밥', '저녁', '점심', '아침', '밥먹',
            '영화', '영화봐', '영화보자',
            '운동', '운동하자',
            # 상태 묻기
            '뭐해', '뭐하고있어', '여기니',
            '어디야', '어디니', '지금뭐해', '뭐하냐',
            # 요청/부탁
            '도와줄래', '도와줄', '부탁', '부탁해',
            '줄래', '줄래요', '보내줄래',
            # 학교/교육
            '학교', '숙제', '공부', '시험', '시험공부',
            '학원', '수업', '개학', '방학',
            # 생활 관련
            '집', '집에', '귀가', '들어가', '돌아와',
            '늦어', '빨리와', '빨리',
            # 기념일
            '생일', '생일축하', '생일이', '기념일',
            '초대', '꼭와', '약속',
            # 생활 변화
            '취업', '면접', '결혼', '이사', '전직',
            '나가', '나갈래', '나간다',
        ],
        'weight': 1.3,  # 0.9 → 1.3 (더 강화)
    },
    1: {  # FINANCE
        'keywords': [
            # 거래 관련 (핵심)
            '입금', '출금', '송금', '이체',
            '결제', '결제완료', '결제안내',
            '잔액', '잔액조회', '계좌',
            # 카드 관련
            '카드', '신용카드', '체크카드',
            '카드사', '카드결제', '카드이용',
            # 은행 (구체적)
            'KB국민', '신한', '하나', '우리', '국민',
            '농협', '농협은행', '기업', '토스',
            '카카오뱅크', '현대카드', '삼성카드',
            # 금리/대출
            '대출', '대출금', '이자', '이자비용',
            '환율', '환율변동',
            # 구독/자동이체
            '자동이체', '이체완료',
            '할부', '일시불', '수수료',
            # 금융상품
            '정기예금', '예금', '적금', '저축',
            '주식', '투자', '증권', '펀드',
            # 보험
            '공과금', '요금', '납부', '납부완료',
            '이용료', '손수료',
            # 거래 확인
            '청구', '명세', '고지', '고지서',
            # 이상거래
            '이상거래', '부정거래', '의심거래',
            '한도', '한도초과',
        ],
        'weight': 0.5,
    },
    2: {  # DELIVERY
        'keywords': [
            # 배송 상태 (강화)
            '배송', '배송시작', '배송중', '배송완료',
            '배송예정', '배송조회', '배송추적', '배송지연',
            '배송기사', '배송료',
            '배달', '배달완료', '배달중', '배달원',
            # 택배사
            '택배', '택배사', '롯데', '한진', 'CJ대한',
            '우체국', '우편', '로젠', '경동',
            # 수령/반품
            '수령', '수령가능', '수령확인',
            '반품', '반품요청', '반품처리', '교환',
            '환불', '환불완료', '환불처리',
            # 배송방식
            '로켓배송', '새벽배송', '당일배송', '드림배송',
            '묶음배송', '무료배송', '배송비',
            # 상세 정보
            '송장', '송장번호', '도착예정', '도착',
            '픽업', '수거', '집화', '분류',
            '소포', '상자', '패키지',
            # 특수 배송
            '국제배송', '해외배송', '통관', '통관중',
            '특송', '도어투도어', '등기우편',
            # 배송 상황
            '부재중', '경비실', '현관', '문 앞',
            '보관', '보관함', '재배송', '배송불가',
            # 업체명
            '쿠팡', 'G마켓', '11번가', '마켓컬리', 'SSG',
            '이마트몰', 'GS25', '편의점',
        ],
        'weight': 1.2,  # 1.0 → 1.2 (강화)
    },
    3: {  # GOVERNMENT
        'keywords': [
            # 정부 기관 (강화)
            '국세청', '경찰청', '검찰청', '행정안전부', '노동청',
            '세무서', '구청', '동주민센터',
            # 공단/보험공단
            '건강보험공단', '국민건강보험', '직장의료보험',
            '교통안전공단', '도로교통공단',
            '근로복지공단',
            # 법무/법원
            '법원', '판결', '소환', '신고', '소장', '고소', '고발',
            # 세금/납부
            '세금', '세무', '과세', '납세',
            '과태료', '벌금', '고지서', '고지',
            '환급금', '환급', '세액공제',
            # 관세/통관
            '관세청', '통관', '수입금지', '수출금지',
            # 재난/안전
            '기상청', '지진', '태풍', '호우', '주의보', '경보',
            '미세먼지', '초미세먼지',
            '소방청', '화재', '불조심',
            '재난문자', '긴급재난',
            # 교육
            '교육청', '학교', '개학', '방학', '학사일정',
            '교육부',
            # 복지/사회보장
            '복지', '사회복지', '기초생활수급', '실업급여',
            '장애인', '노인복지',
            # 병역/국방
            '국방부', '입영', '병역', '징병', '제대',
            # 환경
            '환경부', '환경청', '수질오염', '대기오염',
            # 기타 공공기관
            '정부24', '주민센터', '민원',
            '청약', '공공',
            # 고용/채용
            '고용노동부', '고용센터', '채용', '공채', '임용고시',
            # 통계/금융
            '통계청', '한국은행',
            # 관광
            '관광공사', '여행',
            # 의료/건강
            '보건소', '건강검진', '예방접종', '백신',
            # 관련 표현
            '안내', '신청', '접수', '발급', '처리', '완료',
        ],
        'weight': 2.8,  # 2.5 → 2.8 (강화)
    },
    4: {  # PROMOTION
        'keywords': [
            # 광고 표준 키워드
            '광고', '무료수신거부', '080-',
            # 할인 관련 (강화)
            '할인', '할인권', '할인율', '특가', '가격인하',
            '할인가', '인하', '낮춤', '저가', '초저가',
            # 이벤트/행사
            '이벤트', '행사', '세일', '프로모션', '캠페인',
            '특가행사', '플래시세일', '시즌세일',
            # 쿠폰/포인트
            '쿠폰', '포인트', '적립', '캐시백', '환급',
            '적립금', '리워드', '보상',
            # 무료/체험
            '무료', '체험', '무료배송', '빠른배송', '무료체험',
            '무료서비스',
            # 신상품/신메뉴 (강화)
            '신메뉴', '신상', '신상품', '출시', '런칭',
            '신제품', '새상품', '새메뉴',
            # 수량 제한 (강화)
            '선착순', '한정', '한정판', '기간한정', '재고한정',
            '한정수량', '수량한정', '재고소진',
            # 묶음/세트
            '1+1', '2+1', '번들', '세트', '패키지',
            '묶음', '조합', '세트상품',
            # 멤버십
            '멤버십', '회원', '가입', '혜택', '멤버십가입',
            '회원가입', '가입축하', '가입혜택',
            # 유명 브랜드
            '네이버쇼핑', '무신사', '올리브영', '배달의민족',
            '스타벅스', '아마존', '쿠팡', 'G마켓', '11번가',
            '마켓컬리', 'SSG', 'GS25', '편의점',
            '배달앱', '배민', '배달',
            # 구매 유도 (강화)
            '지금', '한정', '서두르세요', '빨리', '놓치지마',
            '구매하세요', '지금바로', '지금주문', '주문하세요',
            '가입하세요', '신청하세요',
            # 증정/사은품
            '증정', '사은품', '선물', '경품',
            '증정품', '기프트',
            # 예매/예약
            '예매', '예약', '미리구매', '사전주문',
            '예매하세요', '예약가능',
            # 배송 관련 프로모션
            '무료배송', '배송무료', '배송료무료',
            '빠른배송', '신속배송',
            # 기타 프로모션 표현
            '급격한할인', '즉시할인', '현금할인',
            '첫주문', '신규가입', '추천인', '친구초대',
            '재구매', '단골혜택', '단골할인',
            '핫딜', '데일딜',
        ],
        'weight': 2.8,  # 2.6 → 2.8
    },
    5: {  # AUTH
        'keywords': [
            # 인증번호/코드 (강화)
            '인증번호', '인증코드', 'OTP', 'OTP번호',
            '코드', '컨펌코드', '확인코드', '검증코드',
            '비밀번호', '인증', '인증확인', '인증요청',
            '번호', '숫자',
            # 플랫폼별 인증
            'Apple ID', 'Apple', 'Google', '구글',
            '네이버', '카카오', '카카오톡',
            '페이스북', '인스타그램', '라인',
            'Amazon', '아마존',
            # 금융 인증
            '은행', '공동인증서', '금융인증',
            '신용카드', '클릭인증', '문자인증',
            # 보안 관련 (강화)
            '2단계인증', '2FA', '멀티팩터인증',
            '보안코드', '보안키', '보안',
            '기기로그인', '로그인인증', '로그인확인',
            '새로운기기', '새로운환경', '새기기',
            # 유효성/만료
            '유효', '만료', '기한', '시간초과',
            '5분', '10분', '유효시간', '유효기간',
            # 링크/클릭
            '링크', '클릭', '확인', '인증완료',
            '클릭하세요', '확인하세요',
            # 위험/의심
            '위험', '의심', '비정상',
            '타인', '공유금지', '노출금지', '절대금지',
            # 입력 유도
            '입력', '입력하세요', '입력하시기바랍니다',
            '제출', '전송', '입력바랍니다',
        ],
        'weight': 1.9,  # 1.8 → 1.9
    },
    6: {  # WORK
        'keywords': [
            # 회의 관련 (강화)
            '회의', '미팅', '회의시간', '미팅시간',
            '회의실', '미팅실', '회의장', '미팅장',
            '회의참석', '미팅참석', '회의개최',
            # 직급/조직
            '팀장', '부장', '과장', '대리', '사원',
            '차장', '전무', '상무', '대표이사', '이사',
            '인사팀', '부서', '팀',
            # 업무 관련
            '보고', '보고서', '보고드립니다',
            '자료', '자료공유', '자료첨부', '자료요청',
            '검토', '피드백', '검토부탁', '피드백주세요',
            '진행', '진행중', '진행상황', '진행상황공유',
            # 프로젝트/클라이언트
            '프로젝트', '클라이언트', '거래처',
            '프로젝트진행', '프로젝트시작',
            # 배포/개발 (강화)
            '배포', '배포완료', '배포예정', '라이브배포',
            '릴리즈', '스프린트', '데일리',
            '개발', '개발팀', '코딩',
            # 교육/세미나
            '워크샵', '세미나', '교육', '교육받기',
            '사내교육', '교육프로그램',
            # 급여/보상
            '급여', '급여명세서', '급여발급',
            '명세서', '보너스', '성과급', '인센티브',
            # 휴가 관련
            '반차', '연차', '휴가', '휴가신청',
            '휴가승인', '휴가신청승인',
            # 결재/승인
            '결재', '승인', '제출', '결재요청',
            '승인바랍니다', '결재부탁',
            # 회식/팀빌딩
            '회식', '팀빌딩', '팀워크',
            '회식일정', '회식시간',
            # 공지/안내
            '공지', '공지사항', '안내', '안내드립니다',
            '공지드립니다',
            # 실적/성과
            '실적', '실적보고', '성과', '성과급',
            '목표', '목표달성',
            # 고객/거래
            '고객', '고객문의', '고객답변',
            '거래처', '거래',
            # 요청/협의
            '부탁', '부탁드립니다', '요청', '협의',
            '논의', '협력',
            # 기타
            '긴급', '긴급회의', '긴급공지',
            '이슈', '이슈발생', '대응',
            '종료', '완료', '마무리',
            '스케줄', '일정', '일정조정',
            '특허', '계약서', '계약',
            '인사', '인사발령', '인사이동',
        ],
        'weight': 2.5,  # 2.3 → 2.5 (강화)
    },
}

import os
print('설정 완료')
print(f'카테고리 수: {NUM_LABELS}')
for k, v in CATEGORIES.items():
    print(f'  {k:2d}: {v}')
print(f'\n키워드 부스팅 카테고리: {len(KEYWORD_BOOSTS)}개')
print('\n[조정된 키워드 가중치]')
for idx, boost in KEYWORD_BOOSTS.items():
    keywords_count = len(boost['keywords'])
    print(f'  {CATEGORIES[idx]:<12} weight: {boost["weight"]:.1f}  (keywords: {keywords_count}개)')

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'✅ Tokenizer 로드 완료: {MODEL_NAME}')

class SMSDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

print('✅ SMSDataset 클래스 정의 완료')

## 3. 데이터 업로드 및 확인

In [ ]:
import pandas as pd
import os

from google.colab import files
print('metadata.csv 파일을 업로드하세요')
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]
df = pd.read_csv(CSV_PATH)

print(f'\n✅ 총 {len(df)}건의 데이터가 로드되었습니다.')
print("-" * 30)

print("\n[세부 카테고리별 분포]")
print(df['category'].value_counts())

print("\n[데이터 샘플 확인]")
print(df[['category', 'text']].head())

In [ ]:
category_to_idx = {cat: idx for idx, cat in CATEGORIES.items()}
df['label_idx'] = df['category'].map(category_to_idx)

print("✅ 카테고리 → 숫자 변환 완료")
print(f"Mapping: {category_to_idx}")

## 4. 전처리

URL은 학습 시 완전히 제거합니다.

순수 텍스트 문맥만으로 카테고리를 구분합니다.

In [ ]:
import re
import pandas as pd

URL_RE = re.compile(r'http[s]?://\S+')

def clean_text(text: str) -> str:
    """URL 제거 및 공백 정리"""
    text = URL_RE.sub('', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)

# ── 증강 데이터: 자주 오분류되는 케이스 보완 ─────────────────────────
# "배송" → FINANCE 오분류, "회의"/"미팅" → PERSONAL 오분류 방지
augmented_rows = [
    # ─── PERSONAL 증강 데이터 (30개) ───────────────────────────────────
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '엄마 오늘 집 몇 시에 와?'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '아빠 나 이번 주말에 놀러 가고 싶어'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '언니 수원 가는데 같이 갈래?'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '형 내일 시간 돼? 축구 하자'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '누나 요즘 뭐해? 한 달 안 봤는데'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '동생아 밥 먹었어? 숙제는 했어?'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '할머니 안녕하세요 잘 지내세요?'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '할아버지 다음주에 뵐게요'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '친구 오늘 저녁에 만나자'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '자기 퇴근했어 저녁 뭐 먹을까'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '여보 사랑해 내일도 좋은 하루 보내'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '오빠 생일 축하합니다!'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '미안 어제 실수했어'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '고마워! 나중에 갚을게'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '지금 어디야? 빨리 와'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '여행 다녀왔어? 사진 보여줘'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '엄마 용돈 좀 보내줄 수 있어?'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '이번 주말 꼭 와 약속이야'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '영화표 예매했어 내일 6시에 봐'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '엄마 밥 맛있었어 고마워'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '형이 기다린대 어디야 빨리와'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '오늘 바빠? 내일 만날 수 있어?'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '첫눈이 왔어 운동 가자'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '오늘 영화 재미있었어'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '아이스크림 먹을래 내가 사줄게'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '새로운 번호로 연락합니다'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '숙제 다 했어? 내일 제출이야'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '시험은 어땠어? 잘 봤어?'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '결혼식 초대합니다 4월 15일입니다'},
    {'category': 'PERSONAL', 'label_idx': 0, 'clean_text': '이사 갔어 새 주소로 와'},
    
    # ─── DELIVERY 증강 데이터 (30개) ───────────────────────────────────
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[우체국] 배송이 완료되었습니다. 문 앞 보관함 확인하세요.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[쿠팡] 주문하신 상품 배송이 시작되었습니다. 송장번호를 확인해 주세요.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[CJ대한통운] 배송 예정일은 내일입니다. 집에 안 계시면 경비실에 맡깁니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '배송 중 오류가 발생했습니다. 주소 확인 후 재배송 요청 바랍니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[롯데택배] 배송이 완료되었습니다. 수령 확인 부탁드립니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '배달하러 왔는데 부재중입니다. 경비실에 맡겼습니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[GS25] 편의점 택배 도착. 배달 완료. 픽업 1일 내 수령 바랍니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '고객님의 해외 배송 물품이 통관 절차 진행 중입니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[한진택배] 배송 지연 안내: 기상 악화로 내일 배달 예정입니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[마켓컬리] 새벽 배달 완료. 현관 앞에 보관했습니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[11번가] 배송 조회: 오늘 오후 배달 예정입니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[G마켓] 구매하신 상품 출고 완료. 배송 추적 가능합니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[SSG닷컴] 새벽배송 상품 오늘 6시 도착예정입니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[배민] 음식이 배달 중입니다. 5분 후 도착예정입니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[로젠택배] 택배가 분류 중입니다. 내일 배송 예정.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[이마트몰] 주문하신 물품 출고되었습니다. 송장: 12345'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[당일배송] 오늘 오전 배송완료. 외출 중이면 경비실 참고.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[특송] 중요 서류 배송시작. 서명 후 수령 바랍니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[국제배송] 해외 물품 수입통관 중입니다. 2-3일 소요.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[반품접수] 반품이 접수되었습니다. 수거 예정입니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[교환신청] 교환 상품이 발송되었습니다. 기존상품 수거일정.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[환불완료] 환불이 완료되었습니다. 계좌확인 바랍니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[배송료무료] 3만원 이상 구매하여 배송료가 무료입니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[배송지연] 배송이 지연되었습니다. 최대한 빨리 배달하겠습니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[픽업안내] 편의점에서 수령 가능합니다. 수령처: GS25'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[배송불가] 배송주소가 불명확합니다. 연락주세요.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[배송기사연락] 배송기사가 곧 도착합니다. 현관에 있어주세요.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[로켓배송] 로켓배송으로 오늘 배송완료됩니다.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[묶음배송] 여러 상품이 함께 배송됩니다. 개별배송 보다 저렴.'},
    {'category': 'DELIVERY', 'label_idx': 2, 'clean_text': '[배송조회가능] 배송상황을 앱에서 실시간 조회 가능합니다.'},
    
    # ─── WORK 증강 데이터 (40개) ───────────────────────────────────
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '내일 오전 9시 팀 전체 회의 있습니다. 자료 사전 준비 부탁드립니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '오늘 오후 2시 회의실 B에서 주간 회의입니다. 참석 부탁드립니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '[마케팅팀] 이번 주 금요일 2층 회의실에서 전략 회의 진행합니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '긴급 회의 소집. 30분 내 2층 대회의실 집합 바랍니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '팀장님, 회의 자료 공유드렸습니다. 검토 후 피드백 부탁드립니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '오늘 회의 일정이 변경되어 안내드립니다. 오후 3시로 조정됩니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '[경영지원팀] 이번 주 수요일 임원 회의 일정 공지.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '월간 성과 공유 회의가 내일 오전 10시에 있습니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '[영업팀] 이번 분기 실적 발표 회의 일정 잡혔습니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '팀원 모두 내일 오전 10시 회의 참석 바랍니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '내일 고객사와 미팅 있습니다. 오전 10시 본사 1층 회의실 집합.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '[영업팀] 클라이언트 미팅 자료 오늘 중 이메일 공유 바랍니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '팀장님, 오후 미팅 준비 완료했습니다. 시간 맞춰 이동하겠습니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '거래처 미팅이 10분 늦어질 것 같습니다. 미리 연락드립니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '다음 주 화요일 킥오프 미팅 일정 확인 부탁드립니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '내일 프로젝트 미팅 장소가 변경되었습니다. 3층 대회의실로 와주세요.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '[개발팀] 내일 오전 고객사 미팅 후 결과 공유 회의 예정.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '현재 진행 중인 프로젝트 미팅 관련 자료 첨부했습니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '오늘 오후 파트너사 미팅 일정 취소되었습니다. 다음 주로 연기합니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '미팅 장소가 2층 소회의실로 변경되었습니다. 참고 바랍니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '과장님 보고서 검토 부탁드립니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '프로젝트 진행상황 공유 자료 첨부합니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '급여명세서 발급되었습니다 앱에서 확인하세요.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '[휴가신청] 승인되었습니다 4월 15~18일입니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '[교육안내] 4월 15일 사내교육 참석 부탁드립니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '회의 취소 일정 변경되었습니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '[공지] 사무실 이전 5월 1일입니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '실적 보고 이번 달 목표 달성했습니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '[팀] 회식 금요일 6시 회사 앞입니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '배포 완료 라이브 배포 완료되었습니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '결재 요청 예산안 결재 부탁드립니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '고객 문의 답변 부탁드립니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '프로젝트 종료 수고했습니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '[긴급] 이슈 발생 대응 요청합니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '자료 요청 자료 보내주세요.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '성과급 지급 이번 달 성과급 지급입니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '[공지사항] 복지 정책 변경 안내입니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '스케줄 조정 오늘 스케줄이 변경되었습니다.'},
    {'category': 'WORK', 'label_idx': 6, 'clean_text': '[인사팀] 상반기 인사이동 공지합니다.'},
    
    # ─── GOVERNMENT 증강 데이터 (30개) ───────────────────────────────────
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[국세청] 세금 환급금 50만원이 환급 예정입니다. 계좌이체로 송금됩니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[국세청] 세무서에서 안내입니다. 소득세 신고 기간 내일까지입니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[국세청] 종합소득세 고지서가 발급되었습니다. 납부 기한은 4월 30일입니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[경찰청] 교통법규 위반으로 과태료 50만원이 부과되었습니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[경찰청] 주정차 위반 과태료 10만원 납부 기한 안내.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[기상청] 호우주의보 발령되었습니다. 외출 시 주의하시기 바랍니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[기상청] 태풍 예보입니다. 내일 오후 강풍이 불 예정입니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[기상청] 미세먼지 나쁨 경보. 외출 자제 권고합니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[건강보험공단] 4월 건강보험료 고지서입니다. 납부 기한 내일입니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[건강보험공단] 국민건강보험료 변경 안내입니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[건강보험공단] 직장의료보험 자격 상실 안내.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[교통안전공단] 운전면허증 갱신 기간이 임박했습니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[도로교통공단] 자동차 검사 기간 안내입니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[법원] 법원 출석 통지입니다. 4월 25일 오10시 출석하시기 바랍니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[검찰청] 합의 조건부 기소유예 처분 안내.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[교육청] 새 학기 개학 안내. 3월 1일 개학입니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[교육청] 학사일정 안내입니다. 방학 기간 공지입니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[국방부] 병역신고 안내입니다. 만 18세 이상 신고 대상자.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[노동청] 최저임금 인상 안내. 2026년 최저임금 변경입니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[고용노동부] 실업급여 신청 안내입니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[소방청] 화재주의 당부입니다. 불조심하시기 바랍니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[재난정보] 긴급대피 안내입니다. 즉시 대피하시기 바랍니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[보건소] 무료 건강검진 안내입니다. 검진기간 3월~5월.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[보건소] 예방접종 무료 안내입니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[보건소] 코로나 백신 접종 예약 안내입니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[관세청] 통관료 납부 안내입니다. 해외배송 물품 통관료 10,000원.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[환경부] 환경오염 신고입니다. 불법폐기물 신고 1234-5678.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[주민센터] 민원 안내입니다. 등본 발급 신청 가능합니다.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[구청] 행정 안내입니다. 청년 주택 지원금 신청 기간.'},
    {'category': 'GOVERNMENT', 'label_idx': 3, 'clean_text': '[통계청] 인구조사 참여 안내입니다. 조사원 방문 예정.'},
    
    # ─── AUTH 증강 데이터 (30개) ───────────────────────────────────
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[네이버] 인증번호 583920입니다. 5분 내 입력하세요.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[카카오] 인증번호 [847291]입니다. 타인과 공유하지 마세요.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[카카오톡] 인증번호 472015 입력해주세요.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[KB국민은행] OTP 인증번호: 482910입니다.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[신한은행] 보안코드 인증 123456을 입력하세요.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[하나은행] 인증번호 654321입니다. 유효시간 10분.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[우리은행] OTP 코드 789012를 입력해주세요.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[Apple ID] 인증 요청입니다. 확인 코드는 456789입니다.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[Google] 확인 코드: 234567 Google 계정 로그인입니다.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[페이스북] Facebook 로그인 확인 코드: 567890'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[클릭인증] 신한카드 인증번호 890123 입력하시기바랍니다.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[문자인증] 보안코드 345678이 발급되었습니다.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[쿠팡] 회원가입 인증번호 901234입니다.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[배민] 배달의민족 인증 234567 회원가입 확인.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[라인] LINE 새 기기 로그인 인증번호 612345입니다.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[카카오뱅크] 공동인증 인증번호 789012 입력하세요.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[공인인증서] 보안인증 확인 코드 432109입니다.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[2단계인증] 2FA 인증번호 567834입니다.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[보안강화] 새로운 기기에서 로그인되었습니다. 본인 확인 필요.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[로그인확인] 새 환경에서 접속되었습니다. 확인 코드 345678.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[비밀번호변경] 비밀번호 초기화 인증번호 654123입니다.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[계정보안] 의심거래 감지. 즉시 로그인해 확인하세요.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[인증링크] 확인 링크를 클릭하여 인증 완료해주세요.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[이메일인증] 이메일 주소 확인 링크 클릭하세요.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[휴대폰인증] 휴대폰 본인인증 코드 789234입니다.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[신원확인] 신원확인을 위한 인증번호 890123입니다.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[거래확인] 거래 인증번호 123456을 입력해주세요.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[결제인증] 결제 확인 인증번호 234567입니다.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[타인금지] 본 인증번호를 타인과 공유하지 마세요.'},
    {'category': 'AUTH', 'label_idx': 5, 'clean_text': '[만료안내] 인증번호 유효시간이 만료되었습니다. 재발급 요청하세요.'},
    
    # ─── PROMOTION 증강 데이터 (50개 추가) ──────────────────────────────────────
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '(광고) [스타벅스] 새 시즌 음료 출시! 지금 주문하세요. 무료수신거부 080-1234-5678'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '(광고) [배달의민족] 신규가입 축하! 첫주문 3천원 할인. 무료수신거부 080-2345-6789'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '(광고) [무신사] 여름옷 50% 할인 시작! 선착순 한정 특가. 기간한정 080-3456-7890'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '(광고) [쿠팡] 로켓배송 물품 무료배송! 예약 시 빠른배송. 무료수신거부 080-4567-8901'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '(광고) [올리브영] 신상 화장품 1+1 이벤트! 놓치지마세요. 080-5678-9012'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[네이버쇼핑] 관심상품 가격이 인하되었습니다! 특가로 지금 구매하세요.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[G마켓] 플래시세일 진행중! 모든 상품 할인율 최대 70%. 서둘러서 구매하세요.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[11번가] 추천인 이벤트! 친구초대 시 50000원 보상. 놓치지 마세요.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[아마존] 신상품 런칭! 신메뉴 출시 기념 세일. 한정판 빨리 구매하세요.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[SSG닷컴] 멤버십 가입 축하! 가입혜택 10% 할인권 지급.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '시즌세일 시작! 모든 의류 특가 행사. 지금 쇼핑하세요 무료배송.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '100원대 특가 상품! 선착순 수량한정 빠르게 구매하세요.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[편의점택배] 배송료 무료! 모든 상품 무료배송 진행중.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[카드사] 5% 캐시백 이벤트! 결제 시 즉시할인 및 포인트 적립.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[마켓컬리] 신규가입 1000포인트 적립! 첫주문에 사용 가능.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '핫딜! 인기 상품 50% 할인. 재고 소진 시까지 한정 판매.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[배달앱] 오늘의 특가! 음식점별 할인권 20~50% 쿠폰 지급.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '단골 할인! 재구매 고객 10% 추가할인쿠폰 발급.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[영화관] 예매 할인 2천원 할인쿠폰 받으세요. 주말 선예매 이벤트.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[숙박앱] 전국 호텔 30% 할인! 휴가시즌 선예약 이벤트 진행.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[피자헛] 신메뉴 피자 50% 할인! 2판 시 1판 무료 프로모션.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[스포츠브랜드] 새 컬렉션 출시! 신상품 3천원 할인 쿠폰.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '미스터 도넛 1+1! 신상 도넛 두 개 사면 한 개 무료.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[보험사] 가입 이벤트! 신규고객 보장료 20% 할인 행사.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[VIP클럽] 멤버십 등급별 즉시할인! 구매 시 5~15% 자동할인.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[세차장] 신규가입 회원 세차료 50% 할인권 증정!'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '날씨 좋은 날 특가! 오늘 주문 주말 배송료 무료 프로모션.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[PC방] 시간권 30% 할인쿠폰! 추천인 이벤트 기간한정.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[노래방] 신규 회원 시간권 무료체험 권! 친구초대 시 보상금.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[전자제품] 신제품 출시 기념! 얼리버드 특가 한정수량 판매.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[통신사] 신규계약 기변 이벤트! 폰값 할인 및 포인트 적립.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[렌터카] 장기렌트 특가! 월 50만원대 초저가 프로모션.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[패션] 봄신상 런칭! 모든 상품 5천원 이상 할인행사.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[카페] 사이즈업 무료! 기간한정 세트메뉴 1+1 이벤트.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[병원] 특가 검진패키지! 신규고객 40% 할인 기한 임박.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[가구점] 신상 가구 세일! 구매 시 무료배송 및 설치 서비스.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[항공사] 조기예매 특가표! 국제선 편도 특가 한정 판매.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[보험] 실손보험 가입 100만원 캐시백! 지금바로 신청.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[뷰티] 스킨케어 세트 특가! 3개 세트 구성 2개 가격으로 판매.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[음식점] 신메뉴 런칭 기념 첫주문 50% 할인!'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[쇼핑몰] 봄맞이 대세일 진행! 전 상품 20~70% 할인.'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[게임] 신규 가입 축하 골드 10000개 증정!'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[선물세트] 부모님 선물 추천 세트 20% 할인!'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[구독] 정기배송 서비스 첫 달 50% 할인!'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[헬스장] 회원권 가입 첫 달 무료 + 1개월 할인!'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[유학원] 등록금 할인 이벤트 기간한정!'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[렌탈] 신규 고객 초기비 50% 할인!'},
    {'category': 'PROMOTION', 'label_idx': 4, 'clean_text': '[이사] 이삿짐 서비스 예약 고객 10% 할인!'},
]

aug_df = pd.DataFrame(augmented_rows)
df = pd.concat([df, aug_df], ignore_index=True)
print(f'✅ 증강 데이터 {len(augmented_rows)}건 추가. 총 {len(df)}건')

sample = df[df['text'].notna() & df['text'].str.contains('http', na=False)].head(3)
for _, row in sample.iterrows():
    print(f'원문 : {row["text"]}')
    print(f'처리후: {row["clean_text"]}')
    print()



In [ ]:
from sklearn.model_selection import train_test_split

tr, te = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df['label_idx']
)

val_ratio_in_train = VAL_SIZE / (1.0 - TEST_SIZE)
tr, va = train_test_split(
    tr,
    test_size=val_ratio_in_train,
    random_state=SEED,
    stratify=tr['label_idx']
)

has_clean_text = 'clean_text' in tr.columns
print()
print('Train/Val/Test split complete:')
print(f'  Train {len(tr)} rows ({len(tr)/len(df):.1%})')
print(f'  Val   {len(va)} rows ({len(va)/len(df):.1%})')
print(f'  Test  {len(te)} rows ({len(te)/len(df):.1%})')
print(f'train has clean_text column: {has_clean_text}')


In [ ]:
def make_loader(data, shuffle=False, batch_size=BATCH_SIZE):
    ds = SMSDataset(data['clean_text'].tolist(), data['label_idx'].tolist(), tokenizer)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY and torch.cuda.is_available()
    )

tr_loader = make_loader(tr, shuffle=True, batch_size=BATCH_SIZE)
va_loader = make_loader(va, batch_size=EVAL_BATCH_SIZE)
te_loader = make_loader(te, batch_size=EVAL_BATCH_SIZE)

print('DataLoader ready')
print(f'  - train steps per epoch: {len(tr_loader)}')
print(f'  - val steps per epoch:   {len(va_loader)}')
print(f'  - test steps per epoch:  {len(te_loader)}')


## 6. 모델 정의

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

class SMSClassifier(nn.Module):
    """
    개선된 분류기:
    - CLS 토큰 + 마스킹 평균 풀링(Mean Pooling) 결합 → 더 풍부한 문맥 표현
    - Multi-Sample Dropout → 다양한 드롭아웃 비율로 앙상블 효과, 정규화 강화
    """
    def __init__(self, model_name, num_labels, dropout_rate=0.3, num_dropouts=5):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.num_dropouts = num_dropouts

        # 드롭아웃 비율을 0.30 ~ 0.30+0.04*(n-1) 로 분산
        self.dropouts = nn.ModuleList([
            nn.Dropout(dropout_rate + i * 0.04) for i in range(num_dropouts)
        ])

        # CLS + 평균 풀링 결합 (h*2) → 투영 레이어 → 분류 헤드
        self.proj = nn.Sequential(
            nn.Linear(h * 2, h),
            nn.GELU(),
        )
        self.classifier = nn.Linear(h, num_labels)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = out.last_hidden_state  # (B, L, H)

        # CLS 토큰
        cls = last_hidden[:, 0, :]

        # 마스킹 평균 풀링: 패딩 토큰 제외 평균
        mask_exp = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        mean_pool = torch.sum(last_hidden * mask_exp, 1) / \
                    torch.clamp(mask_exp.sum(1), min=1e-9)

        # CLS + 평균 풀링 결합
        pooled = torch.cat([cls, mean_pool], dim=-1)
        h_proj = self.proj(pooled)

        # Multi-Sample Dropout: 각 드롭아웃 마스크의 로짓 평균
        logits = sum(self.classifier(drop(h_proj)) for drop in self.dropouts) \
                 / self.num_dropouts
        return logits


MODEL_NAME = 'klue/bert-base'
NUM_LABELS  = len(CATEGORIES)
DROPOUT     = 0.3
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = SMSClassifier(MODEL_NAME, NUM_LABELS, DROPOUT).to(device)

print(f'모델 로드 완료 | device: {device}')
print(f'파라미터 수: {sum(p.numel() for p in model.parameters()):,}')
print(f'  ┣ CLS + Mean-Pooling 결합')
print(f'  ┗ Multi-Sample Dropout (x{model.num_dropouts})')


## 7. 학습

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm
from collections import Counter
from sklearn.metrics import f1_score
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.label_smooth = label_smoothing

    def forward(self, inputs, targets):
        ce = F.cross_entropy(
            inputs,
            targets,
            weight=self.weight,
            label_smoothing=self.label_smooth,
            reduction='none'
        )
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

class_counts = Counter(tr['label_idx'].tolist())
total_samples = len(tr)
num_classes = len(CATEGORIES)

computed_weights = torch.tensor([
    total_samples / (class_counts.get(i, 1) * num_classes)
    for i in range(num_classes)
], dtype=torch.float).to(device)
computed_weights = computed_weights / computed_weights.mean()

print('Computed class weights:')
for i, w in enumerate(computed_weights):
    count = class_counts.get(i, 0)
    print(f"  {CATEGORIES[i]:<10} (n={count:4d}): {w:.4f}")

criterion = FocalLoss(
    weight=computed_weights,
    gamma=FOCAL_GAMMA,
    label_smoothing=LABEL_SMOOTHING
)

LR_ENCODER = 8e-6
LR_HEAD = 4e-5

optimizer = AdamW([
    {'params': model.encoder.parameters(), 'lr': LR_ENCODER, 'weight_decay': 0.01},
    {'params': model.proj.parameters(), 'lr': LR_HEAD, 'weight_decay': 0.01},
    {'params': model.classifier.parameters(), 'lr': LR_HEAD, 'weight_decay': 0.01},
    {'params': model.dropouts.parameters(), 'lr': LR_HEAD, 'weight_decay': 0.0},
])

total_steps = len(tr_loader) * EPOCHS // GRAD_ACCUM_STEPS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

def evaluate(loader):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for b in loader:
            ids = b['input_ids'].to(device)
            mask = b['attention_mask'].to(device)
            labels = b['label'].to(device)
            logits = model(ids, mask)
            total_loss += criterion(logits, labels).item()
            preds_all.extend(logits.argmax(-1).cpu().tolist())
            labels_all.extend(labels.cpu().tolist())
    acc = sum(p == l for p, l in zip(preds_all, labels_all)) / len(labels_all)
    macro_f1 = f1_score(labels_all, preds_all, average='macro')
    return total_loss / len(loader), acc, macro_f1, preds_all, labels_all

best_val_f1 = 0.0
patience = 4
patience_counter = 0

print(f'Gradient accumulation: {GRAD_ACCUM_STEPS} step')
print(f'Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}')
print(f'Total optimizer steps: {total_steps}')

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad()

    for step, b in enumerate(tqdm(tr_loader, desc=f'Epoch {epoch}/{EPOCHS}'), start=1):
        ids = b['input_ids'].to(device)
        mask = b['attention_mask'].to(device)
        labels = b['label'].to(device)

        logits = model(ids, mask)
        raw_loss = criterion(logits, labels)
        loss = raw_loss / GRAD_ACCUM_STEPS
        loss.backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(tr_loader):
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        tr_loss += raw_loss.item()
        correct += (logits.argmax(-1) == labels).sum().item()
        total += labels.size(0)

    va_loss, va_acc, va_f1, _, _ = evaluate(va_loader)
    print(
        f'[Epoch {epoch:02d}] '
        f'train_loss: {tr_loss/len(tr_loader):.4f}  '
        f'train_acc: {correct/total:.4f}  |  '
        f'val_loss: {va_loss:.4f}  '
        f'val_acc: {va_acc:.4f}  '
        f'val_macro_f1: {va_f1:.4f}'
    )

    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        patience_counter = 0
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'  Best saved (val_macro_f1={va_f1:.4f})')
    else:
        patience_counter += 1
        print(f'  No improvement ({patience_counter}/{patience})')

    if patience_counter >= patience:
        print()
        print(f'Early stopping (best_val_macro_f1={best_val_f1:.4f})')
        break


## 8. 테스트셋 최종 평가

In [ ]:
from sklearn.metrics import classification_report, f1_score

model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
_, te_acc, te_f1, te_preds, te_labels = evaluate(te_loader)

target_names = [CATEGORIES[i] for i in range(len(CATEGORIES))]
print(f'Test Accuracy: {te_acc:.4f}')
print(f'Test Macro F1: {te_f1:.4f}')
print()
print(classification_report(te_labels, te_preds, target_names=target_names, digits=4))


## 9. 카테고리 분류 테스트

학습된 모델이 각 문자를 올바른 카테고리로 분류하는지 확인합니다.

**7개 카테고리(PERSONAL/FINANCE/DELIVERY/GOVERNMENT/PROMOTION/AUTH/WORK)를
문맥 기반으로 정확히 구분하는지가 핵심입니다.**

In [ ]:
import numpy as np

def predict(text: str, use_keyword_boost: bool = True) -> dict:
    """
    문자 텍스트 → 카테고리 예측

    use_keyword_boost=True (기본):
        KEYWORD_BOOSTS에 등록된 키워드가 텍스트에 포함되면
        해당 카테고리의 로짓에 가중치를 가산 → 오분류 보정
    """
    cleaned = clean_text(text)
    enc = tokenizer(
        cleaned,
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    model.eval()
    with torch.no_grad():
        logits = model(
            enc['input_ids'].to(device),
            enc['attention_mask'].to(device)
        )

    # NumPy로 변환하여 키워드 가중치 직접 가산
    logits_np = logits.squeeze(0).cpu().float().numpy().copy()

    # ── 키워드 가중치 적용 ──────────────────────────────────────────
    if use_keyword_boost:
        for cat_idx, boost_info in KEYWORD_BOOSTS.items():
            for kw in boost_info['keywords']:
                if kw in text or kw in cleaned:
                    logits_np[cat_idx] += boost_info['weight']
                    break   # 카테고리당 한 번만 가산 (중복 방지)

    # Softmax (수치 안정성을 위해 max 차감)
    exp_logits = np.exp(logits_np - logits_np.max())
    probs      = exp_logits / exp_logits.sum()

    pred       = int(np.argmax(probs))
    probs_list = probs.tolist()

    return {
        'label':      pred,
        'category':   CATEGORIES[pred],
        'confidence': round(probs_list[pred] * 100, 1),
        'all_probs':  {CATEGORIES[i]: round(p * 100, 1) for i, p in enumerate(probs_list)},
    }


In [ ]:
test_cases = [
    # ── PERSONAL (0) - 20개 ──
    ('엄마 오늘 집 언제 와?', 0, 'PERSONAL-엄마'),
    ('언니 주말에 만나자', 0, 'PERSONAL-언니'),
    ('형 내일 뭐 해? 같이 밥 먹자', 0, 'PERSONAL-형'),
    ('동생아 밥 먹었어?', 0, 'PERSONAL-동생'),
    ('할머니 안녕하세요 잘 지내셔요?', 0, 'PERSONAL-할머니'),
    ('할아버지 다음주에 뵐게요', 0, 'PERSONAL-할아버지'),
    ('친구 요즘 뭐해? 한 달 안 봤는데', 0, 'PERSONAL-친구'),
    ('자기 퇴근했어? 저녁 뭐 먹고 싶어?', 0, 'PERSONAL-배우자'),
    ('새로운 번호로 연락합니다', 0, 'PERSONAL-신번호'),
    ('오빠 생일 축하합니다!', 0, 'PERSONAL-생일'),
    ('누나 어제는 왜 안 받아?', 0, 'PERSONAL-누나'),
    ('미안 어제 실수했어', 0, 'PERSONAL-사과'),
    ('고마워! 나중에 갚을게', 0, 'PERSONAL-감사'),
    ('지금 어디야? 빨리 와', 0, 'PERSONAL-호출'),
    ('여행 다녀왔어? 잘 됐어?', 0, 'PERSONAL-안부'),
    ('아빠 용돈 좀 보내줄 수 있어?', 0, 'PERSONAL-부탁'),
    ('이번 주말 꼭 와! 약속이야', 0, 'PERSONAL-약속'),
    ('영화표 예매했어! 내일 6시 봐', 0, 'PERSONAL-약속'),
    ('엄마 밥 맛있었어 고마워', 0, 'PERSONAL-감사'),
    ('형이 기다린대 어디야?', 0, 'PERSONAL-호출'),

    # ── FINANCE (1) - 20개 ──
    ('[신한은행] 입금 천만원이 입금되었습니다', 1, 'FINANCE-입금'),
    ('[국민은행] 출금 50만원 출금되었습니다. 잔액 234만원', 1, 'FINANCE-출금'),
    ('[삼성카드] 구매 15만원 이용처 스타벅스', 1, 'FINANCE-카드이용'),
    ('[현대카드] 결제대금안내 총액 85만원 결제일 15일', 1, 'FINANCE-결제안내'),
    ('[우리은행] 이체완료 5만원을 송금했습니다', 1, 'FINANCE-송금'),
    ('[토스] 송금 3만원 받았습니다', 1, 'FINANCE-받음'),
    ('[카카오뱅크] 자동이체 전기요금 13만원 납부완료', 1, 'FINANCE-자동이체'),
    ('[롯데카드] 포인트 100포인트 적립되었습니다', 1, 'FINANCE-포인트'),
    ('[KB국민은행] 대출상담 대출가능액 5천만원입니다', 1, 'FINANCE-대출'),
    ('[하나은행] 정기예금 이자 5천원이 입금되었습니다', 1, 'FINANCE-이자'),
    ('[NH농협] 결제실패 카드를 다시 등록해주세요', 1, 'FINANCE-오류'),
    ('[신한카드] 한도초과 사용한도를 초과했습니다', 1, 'FINANCE-한도'),
    ('[기업은행] 명세서 월간 명세서가 발급되었습니다', 1, 'FINANCE-명세서'),
    ('[경주은행] 금리변동 이율이 3.5%로 인상되었습니다', 1, 'FINANCE-금리'),
    ('[대우증권] 배당금 배당금 2만원이 입금되었습니다', 1, 'FINANCE-배당'),
    ('[삼성카드] 할부결제 휴대폰 12개월 할부 승인', 1, 'FINANCE-할부'),
    ('[국민은행] 환율 달러 환율이 1350원으로 하락', 1, 'FINANCE-환율'),
    ('[현대카드] 보상 포인트 프로그램 가입완료', 1, 'FINANCE-프로그램'),
    ('[신한은행] 보험료 의료보험 12만원 납부', 1, 'FINANCE-보험'),
    ('[우리은행] 수수료 송금수수료 1000원 차감', 1, 'FINANCE-수수료'),

    # ── DELIVERY (2) - 20개 ──
    ('[롯데택배] 배송시작 송장 1234567890 배송시작', 2, 'DELIVERY-시작'),
    ('[CJ대한통운] 배송중 상품이 배송중입니다', 2, 'DELIVERY-중'),
    ('[우체국택배] 도착예정 내일 오후 1시 배송예정', 2, 'DELIVERY-예정'),
    ('[한진택배] 배송완료 배송이 완료되었습니다', 2, 'DELIVERY-완료'),
    ('[쿠팡] 픽업가능 편의점에서 수령 가능합니다', 2, 'DELIVERY-수령'),
    ('[로젠택배] 부재중 경비실에 보관되었습니다', 2, 'DELIVERY-부재중'),
    ('[G마켓] 반품접수 반품이 접수되었습니다', 2, 'DELIVERY-반품'),
    ('[11번가] 취소처리 주문이 취소되었습니다', 2, 'DELIVERY-취소'),
    ('[마켓컬리] 새벽배송 새벽 6시 도착 예정', 2, 'DELIVERY-새벽'),
    ('[SSG닷컴] 로켓배송 오늘 도착 예정', 2, 'DELIVERY-로켓'),
    ('[이마트몰] 배송중 택배사 변경: 대한통운', 2, 'DELIVERY-변경'),
    ('[배송사] 배송지연 날씨로 인해 1일 지연', 2, 'DELIVERY-지연'),
    ('[택배] 배송불가 주소가 불명확합니다 연락주세요', 2, 'DELIVERY-불가'),
    ('[택배] 도착직전 배송기사가 5분 후 도착합니다', 2, 'DELIVERY-도착'),
    ('[국제배송] 통관진행중 세관 심사중입니다', 2, 'DELIVERY-통관'),
    ('[택배] 수령확인 고객님 수령을 확인했습니다', 2, 'DELIVERY-확인'),
    ('[쿠팡] 환불처리 환불이 완료되었습니다', 2, 'DELIVERY-환불'),
    ('[택배] 배송료안내 배송료 2500원입니다', 2, 'DELIVERY-료'),
    ('[편의점택배] 도움요청 연락처 입력 요망', 2, 'DELIVERY-연락'),
    ('[특송] 배송예약 내일 시간대를 선택하세요', 2, 'DELIVERY-예약'),

    # ── GOVERNMENT (3) - 20개 ──
    ('[국세청] 환급금 50만원이 환급 예정입니다', 3, 'GOVERNMENT-환급'),
    ('[경찰청] 과태료 주정차 50만원 과태료', 3, 'GOVERNMENT-과태료'),
    ('[기상청] 경보 호우주의보 발령되었습니다', 3, 'GOVERNMENT-경보'),
    ('[건강보험공단] 보험료 4월 보험료 12만원', 3, 'GOVERNMENT-보험'),
    ('[교통안전공단] 운전면허 갱신기간 안내', 3, 'GOVERNMENT-면허'),
    ('[법원] 소환장 증거제출 4월 25일', 3, 'GOVERNMENT-소환'),
    ('[검찰청] 기소예정 수사결과 안내', 3, 'GOVERNMENT-검찰'),
    ('[교육청] 개학안내 3월 1일 개학', 3, 'GOVERNMENT-교육'),
    ('[관광공사] 여행정보 관광지 개장 안내', 3, 'GOVERNMENT-관광'),
    ('[보건소] 건강검진 무료 건강검진 안내', 3, 'GOVERNMENT-검진'),
    ('[환경부] 미세먼지 나쁨 외출 자제', 3, 'GOVERNMENT-미세먼지'),
    ('[경찰] 방범 범죄주의 지역 안내', 3, 'GOVERNMENT-방범'),
    ('[관세청] 통관료 통관료 납부 안내', 3, 'GOVERNMENT-통관'),
    ('[노동청] 최저임금 2026년 1만원 적용', 3, 'GOVERNMENT-임금'),
    ('[도청] 공공일자리 채용 50명 모집', 3, 'GOVERNMENT-채용'),
    ('[시청] 주민투표 의견수렴 4월 30일까지', 3, 'GOVERNMENT-투표'),
    ('[소방청] 화재주의 불조심 당부', 3, 'GOVERNMENT-화재'),
    ('[재난정보] 지진 지진 감지 안내', 3, 'GOVERNMENT-지진'),
    ('[공단] 연금 국민연금 조회 가능', 3, 'GOVERNMENT-연금'),
    ('[주민센터] 민원 신청서 접수 안내', 3, 'GOVERNMENT-민원'),

    # ── PROMOTION (4) - 20개 ──
    ('[스타벅스] 신메뉴 벚꽃라떼 출시! 구매하세요', 4, 'PROMOTION-신메뉴'),
    ('무료배송! 3만원 이상 구매 시 배송료 무료', 4, 'PROMOTION-배송료'),
    ('[편의점] 1+1행사 음료 2개 사면 1개 무료', 4, 'PROMOTION-1+1'),
    ('[쇼핑몰] 할인 모든 상품 30% 할인', 4, 'PROMOTION-할인'),
    ('회원가입 축하 10000포인트 지급', 4, 'PROMOTION-회원'),
    ('[카드사] 캐시백 5% 캐시백 지급', 4, 'PROMOTION-캐시백'),
    ('생일축하 생일 축하 쿠폰 20% 할인', 4, 'PROMOTION-생일'),
    ('[마트] 특가 계란 9900원', 4, 'PROMOTION-특가'),
    ('폐지수거 5천원권 쿠폰 드립니다', 4, 'PROMOTION-쿠폰'),
    ('[영화관] 예매 할인 예매 시 2000원 할인', 4, 'PROMOTION-영화'),
    ('PC방 이용권 50% 할인', 4, 'PROMOTION-PC'),
    ('[뷰티] 세트상품 3개 사면 1개 무료', 4, 'PROMOTION-세트'),
    ('휴대폰 기변 이벤트 최신폰 무료', 4, 'PROMOTION-기변'),
    ('[마켓] 여름옷 여름옷 50% 세일', 4, 'PROMOTION-세일'),
    ('포인트 적립 100원 구매 시 10포인트', 4, 'PROMOTION-적립'),
    ('[카페] 아메리카노 3000원 특가', 4, 'PROMOTION-가격'),
    ('추천인 이벤트 친구초대 5만원 보상', 4, 'PROMOTION-추천'),
    ('[숙박] 예약 30% 할인 숙박료', 4, 'PROMOTION-숙박'),
    ('[피부샵] 이벤트 스킨케어 1+1 이벤트', 4, 'PROMOTION-피부'),
    ('무료체험 1개월 무료 이용 가능', 4, 'PROMOTION-체험'),

    # ── AUTH (5) - 20개 ──
    ('[네이버] 인증번호 123456 입력하세요', 5, 'AUTH-네이버'),
    ('[카카오] 인증번호 654321 5분 유효', 5, 'AUTH-카카오'),
    ('[은행] OTP 인증번호 789012 입력', 5, 'AUTH-은행'),
    ('[Apple] Apple ID 인증 코드 456789', 5, 'AUTH-애플'),
    ('[Google] 확인 코드 234567 구글 로그인', 5, 'AUTH-구글'),
    ('[아마존] 인증 코드 567890 아마존', 5, 'AUTH-아마존'),
    ('[쿠팡] 인증 코드 890123 로그인', 5, 'AUTH-쿠팡'),
    ('[네이버페이] 인증 345678 결제확인', 5, 'AUTH-페이'),
    ('[카카오톡] 인증번호 012345 입력', 5, 'AUTH-톡'),
    ('[라인] 인증 코드 678901 가입', 5, 'AUTH-라인'),
    ('[게임] 인증 번호 901234 2FA 인증', 5, 'AUTH-게임'),
    ('[이메일] 인증 링크 전송됨 클릭하세요', 5, 'AUTH-이메일'),
    ('[은행] 비밀번호 변경 인증 234561', 5, 'AUTH-비번'),
    ('[신용카드] 신청 인증 567812 신청', 5, 'AUTH-신청'),
    ('[증권] OTP 인증 012349 거래', 5, 'AUTH-증권'),
    ('[통신사] 인증 890161 본인확인', 5, 'AUTH-통신'),
    ('[결제] 인증 345689 결제 인증', 5, 'AUTH-결제'),
    ('[앱] 로그인 인증 789234 앱 가입', 5, 'AUTH-앱'),
    ('[보안] 인증 612345 보안 강화', 5, 'AUTH-보안'),
    ('[휴가] 인증 234890 휴가폰 인증', 5, 'AUTH-휴가폰'),

    # ── WORK (6) - 20개 ──
    ('[회의실A] 10시 스탠드업 회의 시작합니다', 6, 'WORK-회의'),
    ('과장님 보고서 검토 부탁드립니다', 6, 'WORK-보고'),
    ('[팀장] 오후 3시 팀 미팅 있습니다', 6, 'WORK-미팅'),
    ('프로젝트 진행상황 공유 자료 첨부', 6, 'WORK-진행'),
    ('급여명세서 발급되었습니다 앱에서 확인', 6, 'WORK-급여'),
    ('퇴사 인사 다음 주가 마지막 주입니다', 6, 'WORK-인사'),
    ('[휴가신청] 승인되었습니다 4월 15~18일', 6, 'WORK-휴가'),
    ('[교육] 신청 4월 15일 사내교육', 6, 'WORK-교육'),
    ('클라이언트 미팅 내일 10시 준비 부탁', 6, 'WORK-클라'),
    ('회의 취소 일정 변경되었습니다', 6, 'WORK-취소'),
    ('[공지] 사무실 이전 5월 1일', 6, 'WORK-공지'),
    ('실적 보고 이번 달 목표 달성', 6, 'WORK-실적'),
    ('[팀] 회식 금요일 6시 회사 앞', 6, 'WORK-회식'),
    ('배포 완료 라이브 배포 완료', 6, 'WORK-배포'),
    ('결재 요청 예산안 결재 바랍니다', 6, 'WORK-결재'),
    ('고객 문의 답변 부탁드립니다', 6, 'WORK-고객'),
    ('프로젝트 종료 수고했습니다', 6, 'WORK-종료'),
    ('[긴급] 이슈 발생 대응 요청', 6, 'WORK-긴급'),
    ('자료 요청 자료 보내주세요', 6, 'WORK-자료'),
    ('성과급 지급 이번 달 성과급 지급', 6, 'WORK-성과'),
]

correct, wrong = 0, []

print(f'{"설명":<25} {"정답":<14} {"예측":<14} {"신뢰도":>6}  결과')
print('─' * 78)

for text, gt, desc in test_cases:
    r = predict(text)
    ok = '✅' if r['label'] == gt else '❌'
    if r['label'] == gt:
        correct += 1
    else:
        wrong.append((desc, CATEGORIES[gt], r['category'], r['confidence'], text))
    print(f"{desc:<25} {CATEGORIES[gt]:<14} {r['category']:<14} {r['confidence']:>5}%  {ok}")

print('─' * 78)
print(f'\n정확도: {correct}/{len(test_cases)} ({correct/len(test_cases)*100:.1f}%)')

if wrong:
    print('\n[❌ 오분류 상세]')
    for desc, gt, pred, conf, text in wrong:
        print(f'  • {desc}')
        print(f'    정답: {gt}  →  예측: {pred} ({conf}%)')
        print(f'    문자: {text}')


In [ ]:
detail_cases = [
    # FINANCE
    ('[신한은행] 카드 대금 납부 완료.',                          'FINANCE'),
    ('[삼성카드] 승인안내 50,000원 이용처: 스타벅스',             'FINANCE'),
    ('[삼성카드] 해외승인 $129.99 이용처: AMAZON.COM',           'FINANCE'),
    # DELIVERY
    ('[CJ대한통운] 미수령 등기우편물 반송 처리.',                 'DELIVERY'),
    ('[우체국택배] 택배 배송 시작.',                              'DELIVERY'),
    ('[쿠팡] 배송완료. 문 앞에 물품을 보관하였습니다.',            'DELIVERY'),
    # GOVERNMENT
    ('[국세청] 세금 납부 완료.',                                  'GOVERNMENT'),
    ('[국세청] 세금 환급금 30만원 환급 예정.',                     'GOVERNMENT'),
    ('[경찰청] 교통법규 위반 고지서.',                             'GOVERNMENT'),
    # PERSONAL
    ('영화 표 예매했어! 내일 6시에 보자.',                         'PERSONAL'),
    ('엄마 오늘 저녁 뭐 먹어요?',                                 'PERSONAL'),
    ('나 내일 수원 가는데 같이 갈래?',                             'PERSONAL'),
    # PROMOTION
    ('(광고) [스타벅스] 리워드 2배 적립 이벤트! 무료수신거부 080-220-7718', 'PROMOTION'),
    ('(광고) [배달의민족] 첫 주문 3천원 할인! 무료수신거부 080-365-6101',   'PROMOTION'),
    ('[네이버쇼핑] 관심상품 가격 인하 알림. 지금 구매하세요.',              'PROMOTION'),
    # AUTH
    ('[네이버] 인증번호 583920입니다.',                            'AUTH'),
    ('[카카오톡] 인증번호 [847291]를 입력해 주세요.',               'AUTH'),
    ('[KB국민은행] OTP 인증번호: 482910.',                         'AUTH'),
    # WORK
    ('내일 회의실 A에서 10시 킥오프 미팅 있습니다.',                'WORK'),
    ('[인사팀] 상반기 인사이동 공지.',                              'WORK'),
    ('과장님, 금일 오후 반차 사용하고자 합니다. 결재 부탁드립니다.',  'WORK'),
]

for text, expected in detail_cases:
    r = predict(text)
    ok = '✅' if r['category'] == expected else '❌'
    print(f'\n{ok} 문자: {text}')
    print(f'   예측: {r["category"]} ({r["confidence"]}%)   |   정답: {expected}')
    print('   확률 분포 (상위 4개):')
    for cat, prob in sorted(r['all_probs'].items(), key=lambda x: -x[1])[:4]:
        bar = '█' * int(prob / 4)
        flag = ' ←' if cat == r['category'] else ''
        print(f'     {cat:<14} {prob:>5}% {bar}{flag}')

In [ ]:
text = input('테스트할 문자를 입력하세요: ')
r = predict(text)

print(f'\n카테고리 : {r["category"]}')
print(f'신뢰도   : {r["confidence"]}%')
print('\n전체 확률 분포:')
for cat, prob in sorted(r['all_probs'].items(), key=lambda x: -x[1]):
    bar = '█' * int(prob / 4)
    flag = ' ←' if cat == r['category'] else ''
    print(f'  {cat:<12} {prob:>5}% {bar}{flag}')

## 10. 모델 저장
> 코랩은 세션 종료 시 파일이 삭제되므로 드라이브에 백업하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy(SAVE_PATH, '/content/drive/MyDrive/sms_category_model.pt')
print('✅ 드라이브 저장 완료')